# 🏗️ Notebook 4: Ontology Construction

Build an OWL ontology programmatically from extracted concepts and relations using **Owlready2**.

---

In [ ]:
!pip install -q owlready2 rdflib
!git clone https://github.com/YOUR_USERNAME/ontology-builder.git 2>/dev/null || true
import sys; sys.path.insert(0, 'ontology-builder')

## 1. Build a Simple Ontology from Scratch

In [ ]:
from owlready2 import *

# Create a new ontology
onto = get_ontology('http://example.org/pharma-ontology/')

with onto:
    # Define classes
    class Drug(Thing): pass
    class Disease(Thing): pass
    class Symptom(Thing): pass
    class Analgesic(Drug): pass       # Subclass
    class NSAID(Analgesic): pass       # Sub-subclass
    
    # Define object properties
    class treats(Drug >> Disease): pass
    class causes(Disease >> Symptom): pass
    class relieves(Drug >> Symptom): pass
    
    # Define data properties
    class hasName(Drug >> str, FunctionalProperty): pass
    class hasDosageMg(Drug >> float): pass
    
    # Create instances
    aspirin = NSAID('Aspirin')
    aspirin.hasName = 'Aspirin'
    aspirin.hasDosageMg = [325.0, 500.0]
    
    headache = Disease('Headache')
    pain = Symptom('Pain')
    
    aspirin.treats.append(headache)
    aspirin.relieves.append(pain)
    headache.causes.append(pain)

# Print what we created
print('Classes:', list(onto.classes()))
print('Object Properties:', list(onto.object_properties()))
print('Data Properties:', list(onto.data_properties()))
print('Individuals:', list(onto.individuals()))
print(f'\nAspirin treats: {aspirin.treats}')
print(f'Aspirin is a: {aspirin.is_a}')

## 2. Save and Inspect the Ontology

In [ ]:
import os
os.makedirs('output', exist_ok=True)

onto.save(file='output/pharma_manual.owl', format='rdfxml')
print('Saved to output/pharma_manual.owl')

# Show the raw OWL/XML
with open('output/pharma_manual.owl') as f:
    content = f.read()
print(f'\nFile size: {len(content)} bytes')
print('\n--- First 1500 chars ---')
print(content[:1500])

## 3. Build Ontology from Extracted Data

Use the `OntologyBuilder` class to automate construction from concepts and relations.

In [ ]:
from src.ontology_builder import OntologyBuilder
from src.concept_extractor import Concept
from src.relation_extractor import Relation

# Simulated extracted data
concepts = [
    Concept(name='Drug', label='Drug'),
    Concept(name='Disease', label='Disease'),
    Concept(name='Symptom', label='Symptom'),
    Concept(name='Analgesic', label='Analgesic', parent='Drug'),
    Concept(name='NSAID', label='NSAID', parent='Analgesic'),
    Concept(name='Opioid', label='Opioid', parent='Analgesic'),
    Concept(name='Headache', label='Headache', parent='Disease'),
    Concept(name='Migraine', label='Migraine', parent='Headache'),
    Concept(name='Pain', label='Pain', parent='Symptom'),
    Concept(name='Inflammation', label='Inflammation', parent='Symptom'),
]

relations = [
    Relation(subject='NSAID', predicate='treats', object='Inflammation'),
    Relation(subject='Analgesic', predicate='relieves', object='Pain'),
    Relation(subject='Headache', predicate='causes', object='Pain'),
    Relation(subject='Migraine', predicate='causes', object='Pain', relation_type='object_property'),
    Relation(subject='Migraine', predicate='isA', object='Headache', relation_type='is_a'),
]

# Build
builder = OntologyBuilder(
    iri='http://example.org/pharma',
    name='PharmaOntology'
)
builder.add_concepts(concepts)
builder.add_relations(relations)

# Add data properties
builder.add_data_properties([
    {'concept': 'Drug', 'attribute': 'hasGenericName', 'value_type': 'string'},
    {'concept': 'Drug', 'attribute': 'hasDosageMg', 'value_type': 'float'},
])

# Stats
stats = builder.get_stats()
print('Ontology Statistics:')
for k, v in stats.items():
    print(f'  {k}: {v}')

# Save
builder.save('output/pharma_auto.owl')
print('\nSaved to output/pharma_auto.owl')

## 4. Build from LLM Output

In [ ]:
from src.llm_extractor import LLMKnowledge

# Simulated LLM output (in production, this comes from llm_extractor.py)
llm_knowledge = LLMKnowledge(
    concepts=[
        {'name': 'Vehicle', 'label': 'Vehicle', 'parent': '', 'definition': 'A means of transport'},
        {'name': 'Car', 'label': 'Car', 'parent': 'Vehicle', 'definition': 'A four-wheeled motor vehicle'},
        {'name': 'ElectricCar', 'label': 'Electric Car', 'parent': 'Car', 'definition': 'A car powered by electricity'},
        {'name': 'Engine', 'label': 'Engine', 'parent': '', 'definition': 'A machine that converts energy'},
        {'name': 'Battery', 'label': 'Battery', 'parent': '', 'definition': 'An energy storage device'},
        {'name': 'Manufacturer', 'label': 'Manufacturer', 'parent': '', 'definition': 'A company that makes vehicles'},
    ],
    relations=[
        {'subject': 'Car', 'predicate': 'hasEngine', 'object': 'Engine'},
        {'subject': 'ElectricCar', 'predicate': 'hasBattery', 'object': 'Battery'},
        {'subject': 'Manufacturer', 'predicate': 'produces', 'object': 'Vehicle'},
    ],
    attributes=[
        {'concept': 'Car', 'attribute': 'hasColor', 'value_type': 'string'},
        {'concept': 'Car', 'attribute': 'hasYear', 'value_type': 'integer'},
        {'concept': 'Battery', 'attribute': 'hasCapacityKwh', 'value_type': 'float'},
    ]
)

builder2 = OntologyBuilder(iri='http://example.org/vehicles')
builder2.from_llm_output(llm_knowledge)

stats2 = builder2.get_stats()
print('Vehicle Ontology Stats:')
for k, v in stats2.items():
    print(f'  {k}: {v}')

builder2.save('output/vehicles.owl')
print('\nSaved to output/vehicles.owl')

## 5. Inspect with rdflib

In [ ]:
from rdflib import Graph

g = Graph()
g.parse('output/pharma_auto.owl', format='xml')

print(f'Total triples: {len(g)}')
print('\nSample triples:')
for i, (s, p, o) in enumerate(g):
    if i >= 15:
        break
    s_short = str(s).split('/')[-1].split('#')[-1]
    p_short = str(p).split('/')[-1].split('#')[-1]
    o_short = str(o).split('/')[-1].split('#')[-1]
    print(f'  {s_short} -- {p_short} --> {o_short}')

---
**Next:** [Notebook 5 — Validation & Querying](05_validation_and_querying.ipynb)